# Pizza Sales Analytics & Business Intelligence

**Purpose:** Reproducible Python analysis supporting the Power BI project.

> Before running this notebook, place the independently sourced and verified CSV dataset in `data/` and update `DATA_PATH` if necessary.

**Important:** Do not use the exact dataset supplied in the internship/masterclasses if the internship instructions prohibit it.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("data")
csv_files = sorted(DATA_DIR.glob("*.csv"))

print("CSV files found:", [p.name for p in csv_files])

if not csv_files:
    raise FileNotFoundError(
        "No CSV file found in data/. Add your verified project dataset first."
    )

DATA_PATH = csv_files[0]
df = pd.read_csv(DATA_PATH)

print("Dataset:", DATA_PATH)
print("Shape:", df.shape)
df.head()


## 1. Data Quality Check

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))

print("\nDuplicate rows:", df.duplicated().sum())


## 2. Standardize Common Column Names

The Power BI file cannot be reliably converted into a Python dataset from the PBIX container alone. Therefore this notebook checks for common pizza-sales field names and reports what is available.

If your dataset uses different names, update the mapping below.


In [ ]:
# Common field-name candidates
candidates = {
    "order_id": ["order_id", "orderid", "order number"],
    "quantity": ["quantity", "qty"],
    "order_date": ["order_date", "date"],
    "order_time": ["order_time", "time"],
    "unit_price": ["unit_price", "price"],
    "total_price": ["total_price", "total", "revenue", "sales"],
    "pizza_name": ["pizza_name", "pizza"],
    "pizza_category": ["pizza_category", "category"],
    "pizza_size": ["pizza_size", "size"],
}

lower_to_original = {c.lower().strip(): c for c in df.columns}

resolved = {}
for logical, options in candidates.items():
    for option in options:
        if option.lower() in lower_to_original:
            resolved[logical] = lower_to_original[option.lower()]
            break

print("Resolved fields:")
for k, v in resolved.items():
    print(f"{k:15} -> {v}")

required_for_basic = ["quantity"]
missing_required = [x for x in required_for_basic if x not in resolved]

if missing_required:
    print("\nBasic KPI calculation requires these fields:", missing_required)
else:
    print("\nBasic quantity field is available.")


## 3. Prepare Date/Time Fields

In [ ]:
# Create useful date/time fields when the corresponding columns exist.
if "order_date" in resolved:
    date_col = resolved["order_date"]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df["year"] = df[date_col].dt.year
    df["month"] = df[date_col].dt.month
    df["month_name"] = df[date_col].dt.month_name()
    df["day_name"] = df[date_col].dt.day_name()

if "order_time" in resolved:
    time_col = resolved["order_time"]
    parsed_time = pd.to_datetime(df[time_col].astype(str), errors="coerce")
    df["hour"] = parsed_time.dt.hour

df.head()


## 4. KPI Analysis

In [ ]:
def numeric_series(logical_name):
    col = resolved.get(logical_name)
    if not col:
        return None
    return pd.to_numeric(df[col], errors="coerce")

quantity = numeric_series("quantity")
total_price = numeric_series("total_price")
unit_price = numeric_series("unit_price")

kpis = {}

if total_price is not None:
    kpis["Total Revenue"] = total_price.sum()

if quantity is not None:
    kpis["Total Pizzas Sold"] = quantity.sum()

if "order_id" in resolved:
    kpis["Total Orders"] = df[resolved["order_id"]].nunique()

if total_price is not None and "order_id" in resolved:
    orders = df[resolved["order_id"]].nunique()
    kpis["Average Order Value"] = total_price.sum() / orders if orders else np.nan

if quantity is not None and "order_id" in resolved:
    orders = df[resolved["order_id"]].nunique()
    kpis["Average Pizzas per Order"] = quantity.sum() / orders if orders else np.nan

pd.Series(kpis, name="Value")


## 5. Category Analysis

In [ ]:
category_col = resolved.get("pizza_category")

if category_col and total_price is not None:
    category_summary = (
        df.assign(_revenue=total_price)
          .groupby(category_col, dropna=False)["_revenue"]
          .agg(["sum", "count"])
          .sort_values("sum", ascending=False)
    )
    category_summary.columns = ["revenue", "rows"]
    display(category_summary)

    category_summary["revenue"].plot(kind="bar", figsize=(9, 5))
    plt.title("Revenue by Pizza Category")
    plt.xlabel("Category")
    plt.ylabel("Revenue")
    plt.tight_layout()
    plt.show()
else:
    print("Category and revenue fields were not both detected.")


## 6. Product Analysis

In [ ]:
product_col = resolved.get("pizza_name")

if product_col and total_price is not None:
    product_summary = (
        df.assign(_revenue=total_price)
          .groupby(product_col, dropna=False)["_revenue"]
          .sum()
          .sort_values(ascending=False)
    )

    print("Top products by revenue:")
    display(product_summary.head(10).to_frame("revenue"))

    print("Products requiring review based on lowest revenue:")
    display(product_summary.tail(10).sort_values().to_frame("revenue"))
else:
    print("Pizza name and revenue fields were not both detected.")


## 7. Time Analysis

In [ ]:
if "year" in df.columns and total_price is not None:
    yearly = df.assign(_revenue=total_price).groupby("year")["_revenue"].sum()
    display(yearly.to_frame("revenue"))

if "day_name" in df.columns and total_price is not None:
    day_summary = (
        df.assign(_revenue=total_price)
          .groupby("day_name")["_revenue"]
          .sum()
          .sort_values(ascending=False)
    )
    display(day_summary.to_frame("revenue"))

if "hour" in df.columns and total_price is not None:
    hour_summary = (
        df.assign(_revenue=total_price)
          .groupby("hour")["_revenue"]
          .sum()
          .sort_values(ascending=False)
    )
    display(hour_summary.to_frame("revenue"))
else:
    print("Date/time fields were not detected.")


## 8. Business Insight Framework

Use the actual outputs above and the Power BI dashboard to document findings in this format:

**Finding → Evidence → Business implication → Action**

Examples of areas to investigate:
- Revenue growth/decline
- High-performing categories
- High/low-performing products
- Pizza-size contribution
- Peak days/hours/day-parts
- Products requiring review
- Opportunities for targeted promotions

Do not enter a claim unless it is supported by the actual dataset.


## 9. Final Validation Checklist

- Dataset source verified
- Internship/masterclass dataset not reused
- Missing values reviewed
- Duplicate rows reviewed
- Required fields confirmed
- KPI results checked against Power BI
- Business findings supported by data
- Recommendations linked to findings
- Notebook runs from top to bottom
